# 04 — Training & evaluation across all split schemes — CIC-IDS-2017 — SVM-RFE study

Identical protocol to the base study: 5 models × schemes `70_30`, `80_20`, `90_10`,
`70_20_10` (train/val/test) and `cv5` (5-fold stratified CV), on the top 40 features
selected by **SVM-RFE**. All splits stratified with the same seed and saved to `splits/`.
Each pipeline scales and SMOTE-balances its own training portion only. A (scheme, model)
whose `metrics.json` already exists is skipped — restarts resume.

In [2]:
DATASET = "cic2017"
FT_ROOT = "ft40_rfe"
TARGET_PER_CLASS = 50_000   # per-class row cap for the in-pipeline under/over sampling
MODELS_TO_RUN = None        # None = all 5; or e.g. ["svm", "knn"]
RUN_CV = True

In [3]:
import sys
from pathlib import Path

def _find_nb_common():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for c in (cand, cand / "notebooks"):
            if (c / "nb_common.py").exists():
                return c
    raise FileNotFoundError("nb_common.py not found relative to " + str(Path.cwd()))
sys.path.insert(0, str(_find_nb_common()))

import time
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import nb_common as nbc
from ids import config
from ids.schema import CANONICAL_COLUMNS
from ids.models.pipelines import build_pipeline
from ids.models.registry import MODEL_NAMES
from ids.evaluation.metrics import evaluate_model
from ids.evaluation.cross_val import cross_validate_models, summarize_cv
from ids.evaluation.plots import plot_confusion_matrix

P = nbc.ft_paths(DATASET, FT_ROOT)
log = nbc.setup_logging(P.logs / "04_experiments.log", "nb04")

df = pd.read_parquet(P.data / "clean.parquet")
X, y = df[CANONICAL_COLUMNS], df["y"].to_numpy()
top40 = nbc.load_json(P.features / "top40.json")["selected"]
mapping = nbc.load_json(P.data / "label_mapping.json")
class_names = [c for c, _ in sorted(mapping.items(), key=lambda kv: kv[1])]
n_classes = len(class_names)
models_to_run = MODELS_TO_RUN or list(MODEL_NAMES)
log.info("%d rows | %d selected features | %d classes | models=%s | target_per_class=%d",
         len(X), len(top40), n_classes, models_to_run, TARGET_PER_CLASS)

2026-07-06 13:12:14,025 INFO nb04: 2520798 rows | 40 selected features | 15 classes | models=['decision_tree', 'random_forest', 'svm', 'naive_bayes', 'knn'] | target_per_class=50000


## 1. Build (or load) all split-scheme indices — stratified, fixed seed, saved to `splits/`

In [4]:
idx = np.arange(len(X))

def holdout(test_size):
    tr, te = train_test_split(idx, test_size=test_size, stratify=y,
                              random_state=config.RANDOM_STATE)
    return {"train_idx": tr, "test_idx": te}

def train_val_test():
    tr_val, te = train_test_split(idx, test_size=0.10, stratify=y,
                                  random_state=config.RANDOM_STATE)
    tr, val = train_test_split(tr_val, test_size=2/9, stratify=y[tr_val],
                               random_state=config.RANDOM_STATE)  # 2/9 of 90% = 20%
    return {"train_idx": tr, "val_idx": val, "test_idx": te}

SPLITS = {}
p_7030 = P.splits / "70_30_indices.npz"
SPLITS["70_30"] = dict(np.load(p_7030)) if p_7030.exists() else holdout(0.30)
SPLITS["80_20"] = holdout(0.20)
SPLITS["90_10"] = holdout(0.10)
SPLITS["70_20_10"] = train_val_test()

for scheme, s in SPLITS.items():
    np.savez_compressed(P.splits / f"{scheme}_indices.npz", **s)
    log.info("%s: %s", scheme,
             " / ".join(f"{k.replace('_idx','')}={len(v)}" for k, v in s.items()))

2026-07-06 13:12:17,075 INFO nb04: 70_30: train=1764558 / test=756240
2026-07-06 13:12:18,395 INFO nb04: 80_20: train=2016638 / test=504160
2026-07-06 13:12:19,695 INFO nb04: 90_10: train=2268718 / test=252080
2026-07-06 13:12:20,994 INFO nb04: 70_20_10: train=1764558 / val=504160 / test=252080


## 2. Train + evaluate every (scheme × model)

In [7]:
def run_one(scheme, name, train_idx, test_idx, val_idx=None):
    out_dir = P.runs / scheme / name
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / "metrics.json"
    if metrics_path.exists():
        log.info("SKIP %s/%s — metrics.json exists", scheme, name)
        return nbc.load_json(metrics_path)

    pipeline = build_pipeline(name, top40, target_per_class=TARGET_PER_CLASS)
    t0 = time.perf_counter()
    pipeline.fit(X.iloc[train_idx], y[train_idx])
    fit_seconds = time.perf_counter() - t0
    log.info("%s/%s: fit on %d rows in %.1fs", scheme, name, len(train_idx), fit_seconds)

    record = {"dataset": DATASET, "scheme": scheme, "model": name,
              "n_train": int(len(train_idx)), "n_test": int(len(test_idx)),
              "target_per_class": TARGET_PER_CLASS, "fit_seconds": round(fit_seconds, 3)}
    evals = [("test", test_idx, "")]
    if val_idx is not None:
        record["n_val"] = int(len(val_idx))
        evals.append(("validation", val_idx, "_val"))

    for split_name, eidx, suffix in evals:
        m, _, y_pred, y_proba = evaluate_model(name, pipeline, X.iloc[eidx], y[eidx])
        cm = confusion_matrix(y[eidx], y_pred, labels=np.arange(n_classes))
        np.save(out_dir / f"confusion_matrix{suffix}.npy", cm)
        np.save(out_dir / f"predictions{suffix}.npy", y_pred)
        if y_proba is not None:
            np.save(out_dir / f"probas{suffix}.npy", y_proba.astype(np.float32))
        rep = classification_report(y[eidx], y_pred, labels=np.arange(n_classes),
                                    target_names=class_names, output_dict=True,
                                    zero_division=0)
        pd.DataFrame(rep).T.to_csv(out_dir / f"per_class{suffix}.csv")
        if split_name == "test":
            plot_confusion_matrix(f"{name} ({scheme})", cm, class_names, out_dir)
        record[split_name] = m.to_dict()
        log.info("%s/%s [%s]: acc=%.4f f1_macro=%.4f roc_auc=%s det=%.3fms",
                 scheme, name, split_name, m.accuracy, m.f1_macro,
                 "n/a" if m.roc_auc is None else round(m.roc_auc, 4),
                 m.detection_time_ms)

    nbc.save_json(record, metrics_path)
    return record

records = []
with nbc.timed(log, "all holdout schemes"):
    for scheme, s in SPLITS.items():
        with nbc.timed(log, f"scheme {scheme}"):
            for name in models_to_run:
                records.append(run_one(scheme, name, s["train_idx"], s["test_idx"],
                                       s.get("val_idx")))
pd.DataFrame([{ "scheme": r["scheme"], "model": r["model"],
                "fit_s": r["fit_seconds"], **{k: round(v, 4) if isinstance(v, float) else v
                for k, v in r["test"].items() if k not in ("name",)}} for r in records])

2026-07-06 14:09:19,690 INFO nb04: START all holdout schemes
2026-07-06 14:09:19,692 INFO nb04: START scheme 70_30
2026-07-06 14:09:19,694 INFO nb04: SKIP 70_30/decision_tree — metrics.json exists
2026-07-06 14:09:19,696 INFO nb04: SKIP 70_30/random_forest — metrics.json exists
2026-07-06 14:09:19,697 INFO nb04: SKIP 70_30/svm — metrics.json exists
2026-07-06 14:09:19,699 INFO nb04: SKIP 70_30/naive_bayes — metrics.json exists
2026-07-06 14:09:19,700 INFO nb04: SKIP 70_30/knn — metrics.json exists
2026-07-06 14:09:19,702 INFO nb04: END   scheme 70_30 (0.0s)
2026-07-06 14:09:19,703 INFO nb04: START scheme 80_20
2026-07-06 14:09:19,706 INFO nb04: SKIP 80_20/decision_tree — metrics.json exists
2026-07-06 14:09:19,709 INFO nb04: SKIP 80_20/random_forest — metrics.json exists
2026-07-06 14:09:19,710 INFO nb04: SKIP 80_20/svm — metrics.json exists
2026-07-06 14:09:19,710 INFO nb04: SKIP 80_20/naive_bayes — metrics.json exists
2026-07-06 14:09:23,310 INFO nb04: 80_20/knn: fit on 2016638 rows 

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   11.3s


2026-07-06 14:21:50,261 INFO nb04: 90_10/random_forest: fit on 2268718 rows in 31.9s


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   29.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.4s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.4s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.4s finished


2026-07-06 14:21:52,820 INFO nb04: 90_10/random_forest [test]: acc=0.9953 f1_macro=0.8569 roc_auc=0.9998 det=0.004ms
2026-07-06 14:21:58,613 INFO nb04: 90_10/svm: fit on 2268718 rows in 5.7s
2026-07-06 14:22:00,740 INFO nb04: 90_10/svm [test]: acc=0.7757 f1_macro=0.3696 roc_auc=0.9731 det=0.000ms
2026-07-06 14:22:03,401 INFO nb04: 90_10/naive_bayes: fit on 2268718 rows in 2.7s
2026-07-06 14:22:05,566 INFO nb04: 90_10/naive_bayes [test]: acc=0.7079 f1_macro=0.3680 roc_auc=0.9648 det=0.001ms
2026-07-06 14:22:08,455 INFO nb04: 90_10/knn: fit on 2268718 rows in 2.9s
2026-07-06 14:27:57,938 INFO nb04: 90_10/knn [test]: acc=0.9891 f1_macro=0.7247 roc_auc=0.9979 det=0.455ms
2026-07-06 14:27:57,939 INFO nb04: END   scheme 90_10 (415.4s)
2026-07-06 14:27:57,940 INFO nb04: START scheme 70_20_10
2026-07-06 14:28:11,247 INFO nb04: 70_20_10/decision_tree: fit on 1764558 rows in 13.3s
2026-07-06 14:28:12,297 INFO nb04: 70_20_10/decision_tree [test]: acc=0.9963 f1_macro=0.8132 roc_auc=0.9981 det=0.00

...

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   28.5s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.4s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.4s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.4s finished


2026-07-06 14:28:47,233 INFO nb04: 70_20_10/random_forest [test]: acc=0.9957 f1_macro=0.8544 roc_auc=0.9998 det=0.004ms


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished


2026-07-06 14:28:51,508 INFO nb04: 70_20_10/random_forest [validation]: acc=0.9954 f1_macro=0.8507 roc_auc=0.9998 det=0.003ms
2026-07-06 14:28:57,076 INFO nb04: 70_20_10/svm: fit on 1764558 rows in 5.5s
2026-07-06 14:28:58,692 INFO nb04: 70_20_10/svm [test]: acc=0.7760 f1_macro=0.3669 roc_auc=0.9739 det=0.000ms
2026-07-06 14:29:01,721 INFO nb04: 70_20_10/svm [validation]: acc=0.7754 f1_macro=0.3664 roc_auc=0.9735 det=0.000ms
2026-07-06 14:29:03,788 INFO nb04: 70_20_10/naive_bayes: fit on 1764558 rows in 2.1s
2026-07-06 14:29:06,010 INFO nb04: 70_20_10/naive_bayes [test]: acc=0.7209 f1_macro=0.3729 roc_auc=0.966 det=0.001ms
2026-07-06 14:29:09,268 INFO nb04: 70_20_10/naive_bayes [validation]: acc=0.7207 f1_macro=0.4370 roc_auc=0.9658 det=0.001ms
2026-07-06 14:29:11,362 INFO nb04: 70_20_10/knn: fit on 1764558 rows in 2.1s
2026-07-06 14:35:02,806 INFO nb04: 70_20_10/knn [test]: acc=0.9890 f1_macro=0.7288 roc_auc=0.9979 det=0.461ms
2026-07-06 14:46:55,487 INFO nb04: 70_20_10/knn [validatio

,scheme,model,fit_s,accuracy,precision_macro,recall_macro,f1_macro,roc_auc,detection_time_ms,n_test
0,70_30,decision_tree,24.737,0.9962,0.7378,0.9054,0.7978,0.9981,0.0004,756240
1,70_30,random_forest,39.385,0.9950,0.8118,0.9100,0.8386,0.9998,0.0031,756240
2,70_30,svm,10.775,0.7648,0.3060,0.8689,0.3668,0.9719,0.0014,756240
3,70_30,naive_bayes,2.974,0.7062,0.4021,0.7597,0.4211,0.9651,0.0015,756240
4,70_30,knn,2.243,0.9888,0.6499,0.9118,0.7064,0.9978,0.4613,756240
5,80_20,decision_tree,14.270,0.9965,0.7407,0.9112,0.7959,0.9983,0.0002,504160
6,80_20,random_forest,32.360,0.9957,0.8321,0.9289,0.8594,0.9998,0.0033,504160
7,80_20,svm,6.794,0.7602,0.3041,0.8680,0.3635,0.9721,0.0014,504160
8,80_20,naive_bayes,3.096,0.7078,0.4035,0.7531,0.4117,0.9647,0.0006,504160
9,80_20,knn,3.600,0.9892,0.6544,0.9310,0.7125,0.9977,0.4716,504160


## 3. Five-fold stratified cross-validation (scheme `cv5`)

In [8]:
cv_dir = P.runs / "cv5"
cv_dir.mkdir(parents=True, exist_ok=True)
cv_scores_path = cv_dir / "cv_scores.csv"

if not RUN_CV:
    log.info("RUN_CV=False — skipping")
elif cv_scores_path.exists():
    log.info("SKIP cv5 — cv_scores.csv exists")
    cv_summary = pd.read_csv(cv_dir / "cv_summary.csv")
else:
    with nbc.timed(log, "5-fold stratified CV, all models"):
        scores, _ = cross_validate_models(X, y, top40, model_names=models_to_run,
                                          n_splits=5, target_per_class=TARGET_PER_CLASS)
    scores.to_csv(cv_scores_path, index=False)
    cv_summary = summarize_cv(scores)
    cv_summary.to_csv(cv_dir / "cv_summary.csv", index=False)
cv_summary if RUN_CV else None

2026-07-06 16:14:50,989 INFO nb04: START 5-fold stratified CV, all models
2026-07-06 16:15:06,547 INFO ids.evaluation.cross_val: CV decision_tree fold 1/5: f1_macro=0.7881 acc=0.9958 (13.5s)
2026-07-06 16:15:22,112 INFO ids.evaluation.cross_val: CV decision_tree fold 2/5: f1_macro=0.8188 acc=0.9959 (13.7s)
2026-07-06 16:15:36,724 INFO ids.evaluation.cross_val: CV decision_tree fold 3/5: f1_macro=0.7952 acc=0.9963 (12.9s)
2026-07-06 16:15:52,085 INFO ids.evaluation.cross_val: CV decision_tree fold 4/5: f1_macro=0.7950 acc=0.9957 (13.3s)
2026-07-06 16:16:06,474 INFO ids.evaluation.cross_val: CV decision_tree fold 5/5: f1_macro=0.8007 acc=0.9962 (12.6s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   13.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   35.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished


2026-07-06 16:16:47,150 INFO ids.evaluation.cross_val: CV random_forest fold 1/5: f1_macro=0.8368 acc=0.9957 (36.9s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   14.3s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   36.5s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished


2026-07-06 16:17:28,701 INFO ids.evaluation.cross_val: CV random_forest fold 2/5: f1_macro=0.8250 acc=0.9947 (38.1s)


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   46.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.7min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished


2026-07-06 16:20:07,555 INFO ids.evaluation.cross_val: CV random_forest fold 4/5: f1_macro=0.8660 acc=0.9966 (103.1s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   19.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   47.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.9s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.9s finished


2026-07-06 16:21:00,862 INFO ids.evaluation.cross_val: CV random_forest fold 5/5: f1_macro=0.8463 acc=0.9961 (49.2s)
2026-07-06 16:21:10,380 INFO ids.evaluation.cross_val: CV svm fold 1/5: f1_macro=0.3738 acc=0.7764 (6.3s)
2026-07-06 16:21:20,346 INFO ids.evaluation.cross_val: CV svm fold 2/5: f1_macro=0.3748 acc=0.7785 (6.7s)
2026-07-06 16:21:32,261 INFO ids.evaluation.cross_val: CV svm fold 3/5: f1_macro=0.3626 acc=0.7773 (8.7s)
2026-07-06 16:21:41,842 INFO ids.evaluation.cross_val: CV svm fold 4/5: f1_macro=0.3899 acc=0.7675 (6.2s)
2026-07-06 16:21:54,626 INFO ids.evaluation.cross_val: CV svm fold 5/5: f1_macro=0.3644 acc=0.7762 (9.6s)
2026-07-06 16:21:59,562 INFO ids.evaluation.cross_val: CV naive_bayes fold 1/5: f1_macro=0.4351 acc=0.7167 (1.9s)
2026-07-06 16:22:04,536 INFO ids.evaluation.cross_val: CV naive_bayes fold 2/5: f1_macro=0.4374 acc=0.7078 (1.9s)
2026-07-06 16:22:09,760 INFO ids.evaluation.cross_val: CV naive_bayes fold 3/5: f1_macro=0.4221 acc=0.7112 (2.1s)
2026-07-06 

,model,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,roc_auc_mean,roc_auc_std
0,decision_tree,0.995962,0.000229,0.745327,0.013594,0.913044,0.031040,0.799574,0.011652,0.997953,0.000110
1,knn,0.988876,0.000309,0.663920,0.008478,0.905899,0.015942,0.716417,0.005430,0.997718,0.000101
2,naive_bayes,0.710982,0.003677,0.403576,0.000771,0.764325,0.021671,0.428570,0.009810,0.964707,0.000989
3,random_forest,0.995801,0.000698,0.832464,0.022362,0.902444,0.010119,0.847033,0.016935,0.999793,0.000024
4,svm,0.775205,0.004389,0.311368,0.009057,0.840264,0.005941,0.373111,0.010857,0.972899,0.000866


In [9]:
n_files = sum(1 for _ in P.runs.rglob("*") if _.is_file())
log.info("notebook 04 complete — %d artifact files under %s", n_files, P.runs)

2026-07-06 17:17:25,073 INFO nb04: notebook 04 complete — 142 artifact files under /Users/MAC/Projects/IDS/backend/artifacts/ft40_rfe/cic2017/runs
